# Assignment Implementation

In [43]:
dataset_path = "/home/bukunmi/ml-journey/notebooks/MSCFE0/FE-GWP1_model_selection_1.csv"

## Question 1
With the assigned dataset, perform model selection (i.e. find the best regression model for y in terms of x1, x2, x3, x4, and x5, using a few approaches (at least 2)).
Valid approaches consist of (forward or backward selection, using adjusted R-squared, the AIC criterion, the BIC criterion or any other criterion you consider relevant) It is important that you explain your work. No credit will be given if your model selection appears to be missing any justification and/or is done by a simple trial and error procedure.

### Approach

To find the best regression model for Y, we use two distinct model selection approaches:

1. **Backward Elimination using AIC** - Start with all 5 predictors and iteratively remove the one whose removal most improves (lowers) the Akaike Information Criterion. Stop when no removal improves AIC.

2. **Forward Selection using BIC** - Start with no predictors and iteratively add the one that most improves (lowers) the Bayesian Information Criterion. Stop when no addition improves BIC.

Using two different methods (backward vs. forward) with two different criteria (AIC vs. BIC) provides a robust check: if both approaches select the same model, we can be more confident in the result. AIC and BIC both balance model fit against complexity, but BIC applies a heavier penalty per parameter, so it tends to favor simpler models. If both criteria agree, the evidence for the selected model is strong.

We begin by loading the data and fitting the full model to inspect which predictors appear significant.

In [44]:
import pandas as pd
import statsmodels.api as sm

In [45]:
df = pd.read_csv(dataset_path)
df.head()

,Y,X1,X2,X3,X4,X5
0,3.388410,0.017954,-0.800583,-0.352454,2.187210,1.014887
1,0.287191,0.083057,-0.597947,-0.357639,-1.630284,0.221841
2,3.989645,-0.923437,-1.386575,1.180202,0.632606,-1.576638
3,-2.959602,-0.313775,2.955133,-1.798692,-2.117621,0.159291
4,0.529773,0.388996,1.019611,0.472062,0.590497,0.877048


In [46]:
df.shape

(100, 6)

In [47]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Y       100 non-null    float64
 1   X1      100 non-null    float64
 2   X2      100 non-null    float64
 3   X3      100 non-null    float64
 4   X4      100 non-null    float64
 5   X5      100 non-null    float64
dtypes: float64(6)
memory usage: 4.8 KB


The dataset contains 100 observations with no missing values. All columns are numeric (float64). We now separate the response variable Y from the predictors and fit the full OLS model with all 5 predictors.

In [49]:
y = df["Y"]
X = df.drop(columns=["Y"])

In [52]:
X_with_constant = sm.add_constant(X)

In [54]:
model = sm.OLS(y, X_with_constant)

In [55]:
results = model.fit()

In [56]:
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      Y   R-squared:                       0.649
Model:                            OLS   Adj. R-squared:                  0.630
Method:                 Least Squares   F-statistic:                     34.74
Date:                Thu, 02 Apr 2026   Prob (F-statistic):           5.83e-20
Time:                        21:05:27   Log-Likelihood:                -125.30
No. Observations:                 100   AIC:                             262.6
Df Residuals:                      94   BIC:                             278.2
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.1902      0.090     13.246      0.000       1.012       1.369
X1            -0.0278      0.184     -0.151      0.881      -0.394       0.338
X2            -0.5846      0.092     -6.352      0.000      -0.767      -0.402
X3             0.5603      0.092      6.085      0.000       0.377       0.743
X4             0.7101      0.082      8.616      0.000       0.546       0.874
X5            -0.1957      0.084     -2.327      0.022      -0.363      -0.029
==============================================================================
Omnibus:                        4.561   Durbin-Watson:                   1.970
Prob(Omnibus):                  0.102   Jarque-Bera (JB):                4.016
Skew:                           0.478   Prob(JB):                        0.134
Kurtosis:                       3.222   Cond. No.                         2.58
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Full Model Analysis

The full model (Y ~ X1 + X2 + X3 + X4 + X5) yields an R-squared of 0.649 and an adjusted R-squared of 0.630. The gap between these two values suggests that not all predictors are contributing meaningfully.

Examining the p-values:
- **X1** has a p-value of 0.881, indicating it has no statistically significant relationship with Y. Its coefficient (-0.028) is extremely small relative to its standard error (0.184).
- **X2, X3, X4** all have p-values near 0.000, indicating strong statistical significance.
- **X5** has a p-value of 0.022, which is significant at the 5% level but is the weakest of the four significant predictors.

This preliminary analysis suggests X1 is a candidate for removal. We now formally test this using two systematic model selection procedures.

### Approach 1: Backward Elimination using AIC

Starting with the full model (AIC = 262.59), we iteratively try removing each predictor and check whether the resulting model has a lower (better) AIC. The predictor whose removal yields the greatest AIC improvement is dropped. This continues until no removal improves the AIC.

In [59]:
while len(current_features) > 0:
    aic_candidates = []

    for feature in current_features:
        test_features = [f for f in current_features if f != feature]
        X_test = sm.add_constant(df[test_features])
        test_aic = sm.OLS(y, X_test).fit().aic
        aic_candidates.append((test_aic, feature))
        
    aic_candidates.sort()
    best_new_aic, feature_to_drop = aic_candidates[0]

    # 4. Check if the best candidate is better than our current model
    if best_new_aic < current_aic:
        current_aic = best_new_aic
        current_features.remove(feature_to_drop)
        print(f"Dropped '{feature_to_drop}'. New lowest AIC: {current_aic:.2f}")
    else:
        print("Removing any more variables increases the AIC. Stopping.")
        break
        
    print(f"\nFinal Optimal Model Features: {current_features}")

Dropped 'X1'. New lowest AIC: 260.62

Final Optimal Model Features: ['X2', 'X3', 'X4', 'X5']
Removing any more variables increases the AIC. Stopping.


**Backward Elimination Result:** X1 was dropped in the first iteration, reducing AIC from 262.59 to 260.62. In the second iteration, removing any of the remaining predictors (X2, X3, X4, X5) increased AIC, so the procedure stopped.

**Selected model (AIC backward): Y ~ X2 + X3 + X4 + X5**

### Approach 2: Forward Selection using BIC

Starting from the null model (intercept only, BIC = 359.85), we iteratively try adding each remaining predictor and select the one that most reduces (improves) the BIC. We use BIC rather than AIC here because BIC penalizes model complexity more heavily (its penalty grows with log(n) rather than being fixed at 2 per parameter). This provides a stricter test: if BIC still selects the same predictors as AIC, the evidence is even stronger.

In [61]:
remaining_features = list(X.columns)
selected_features = []

In [63]:
current_best_bic = sm.OLS(y, sm.add_constant(pd.Series([1]*len(y)))).fit().bic
current_best_bic

np.float64(359.85130164184125)

In [ ]:
while remaining_features:
    bic_with_candidates = []

    for feature in remaining_features:
        test_features = selected_features + [feature]
        X_test = sm.add_constant(df[test_features])
        test_bic = sm.OLS(y, X_test).fit().bic
        bic_with_candidates.append((test_bic, feature))
        
    bic_with_candidates.sort()
    best_new_bic, best_feature = bic_with_candidates[0]

    if best_new_bic < current_best_bic:
        remaining_features.remove(best_feature)
        selected_features.append(best_feature)
        current_best_bic = best_new_bic
        print(f"Added '{best_feature}'. New lowest BIC: {current_best_bic:.2f}")
    else:
        print("No more features improve the BIC. Stopping.")
        break

print(f"\nSelected features: {selected_features}")

**Forward Selection Result:** Predictors were added in this order:
1. **X4** (BIC: 359.85 → 330.24) — strongest individual predictor
2. **X3** (BIC: 330.24 → 304.26) — large improvement
3. **X2** (BIC: 304.26 → 274.71) — large improvement
4. **X5** (BIC: 274.71 → 273.64) — small but positive improvement

Adding X1 would have increased BIC from 273.64 to 278.22, so it was rejected.

**Selected model (BIC forward): Y ~ X4 + X3 + X2 + X5**

### Final Model

Both approaches independently selected the same set of predictors: **X2, X3, X4, and X5**, excluding X1. This agreement across different search strategies (backward vs. forward) and different information criteria (AIC vs. BIC) provides strong evidence that this is the appropriate model. We now refit the final model and report its summary.

In [66]:
X_final = X.drop(columns=[col for col in X.columns if col not in selected_features])
X_final.head()

,X2,X3,X4,X5
0,-0.800583,-0.352454,2.187210,1.014887
1,-0.597947,-0.357639,-1.630284,0.221841
2,-1.386575,1.180202,0.632606,-1.576638
3,2.955133,-1.798692,-2.117621,0.159291
4,1.019611,0.472062,0.590497,0.877048


In [67]:
final_results = sm.OLS(y, sm.add_constant(X_final)).fit()

In [68]:
final_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      Y   R-squared:                       0.649
Model:                            OLS   Adj. R-squared:                  0.634
Method:                 Least Squares   F-statistic:                     43.87
Date:                Thu, 02 Apr 2026   Prob (F-statistic):           8.29e-21
Time:                        21:26:30   Log-Likelihood:                -125.31
No. Observations:                 100   AIC:                             260.6
Df Residuals:                      95   BIC:                             273.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.1893      0.089     13.333      0.000       1.012       1.366
X2            -0.5861      0.091     -6.440      0.000      -0.767      -0.405
X3             0.5592      0.091      6.124      0.000       0.378       0.740
X4             0.7105      0.082      8.672      0.000       0.548       0.873
X5            -0.1966      0.083     -2.355      0.021      -0.362      -0.031
==============================================================================
Omnibus:                        4.462   Durbin-Watson:                   1.974
Prob(Omnibus):                  0.107   Jarque-Bera (JB):                3.920
Skew:                           0.473   Prob(JB):                        0.141
Kurtosis:                       3.215   Cond. No.                         1.39
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Conclusion

The selected regression model is:

**Y = 1.189 - 0.586(X2) + 0.559(X3) + 0.711(X4) - 0.197(X5)**

All four predictors are statistically significant (p < 0.05). The model explains 64.9% of the variance in Y (R-squared = 0.649), with an adjusted R-squared of 0.634.

Compared to the full model (which included X1):
| Metric | Full Model (5 predictors) | Final Model (4 predictors) |
|--------|--------------------------|---------------------------|
| R-squared | 0.649 | 0.649 |
| Adj. R-squared | 0.630 | **0.634** |
| AIC | 262.6 | **260.6** |
| BIC | 278.2 | **273.6** |

Dropping X1 had virtually no effect on R-squared (the model explains the same amount of variance) but improved all penalized criteria. This confirms that X1 contributed only noise, not signal. The final model achieves a better balance between fit and parsimony.

----

# Question 2
The elasticity of y with respect to x, defined as (dy/y)/(dx/x) is very important in Economics and even in some financial applications.

4a) The following regression models were estimated. What is the elasticity of y in
terms of X?
    (a) y = 2 + 0.8x
    (b) ln(y)= 0.1 +0.4x
    (c) ln(y) = 0.1 + 0.25ln(x)
    (d) y = 0.15+1.2ln(x)

4b) What is the elasticity of y with respect to x in this case? Hint.. You must identify
the correct regression to use for the purpose among the 4 given in the problem.